# TransferAttack PGN Generation & Evaluation Pipeline

Make sure your runtime is set to **GPU** (Runtime > Change runtime type -> T4 GPU or better).

In [ ]:
!nvidia-smi
import sys
import subprocess
try:
    res = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if res.returncode != 0:
        raise Exception()
except:
    print('!!! NO GPU PHYSICALLY ATTACHED !!!')
    print('Go to Runtime > Change runtime type > confirm T4 GPU is selected, then Runtime > Disconnect and delete runtime, then reconnect and re-run from the top.')
    sys.exit(1)


In [ ]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('GPU Memory growth enabled.')
    except RuntimeError as e:
        print(e)
else:
    tf.config.set_visible_devices([], 'GPU')
    print('Forcing CPU-only mode.')


### Setup Repository and Requirements

In [ ]:
!git clone -b pgn-attack https://github.com/Chidroopakanaparthy/transferattack.git
%cd transferattack
!pip install -r requirements.txt -q

### Extract Dataset and Setup IR152

This cell downloads the dataset using the gdown link provided.
**ATTENTION:** You MUST mount your Google Drive and copy the custom IR152 weights (`IR152.h5`) into the `core/` folder before running the pipeline.

In [ ]:
!gdown 1CD1NjufQJeImCMjZbI_yDDeWWMmRVvR_
!unzip -q dataset_extractedfaces.zip

from google.colab import drive
drive.mount('/content/drive')

# MODIFY THIS PATH to point to your shared drive's IR152.h5 file
!cp "/content/drive/MyDrive/Shared_Drive_Path/IR152.h5" core/IR152.h5


### Dummy IR152 Test Cell

In [ ]:
import os
import numpy as np
import tensorflow as tf

if not os.path.exists('core/IR152.h5'):
    raise FileNotFoundError('IR152.h5 is MISSING! Please fix the copy path above.')
else:
    print('IR152.h5 found. Attempting to load...')
    ir152_model = tf.keras.models.load_model('core/IR152.h5', compile=False)
    dummy_img = np.random.rand(1, 112, 112, 3).astype(np.float32)
    out = ir152_model.predict(dummy_img, verbose=0)
    print(f'IR152 successfully loaded! Output shape: {out.shape}')


### Run Pipeline (Parts A, B, and C)

In [ ]:
# Smoke Test FIRST
!PYTHONPATH=. python scripts/run_pgn_pipeline.py --smoke-test

In [ ]:
# Full Pipeline
!PYTHONPATH=. python scripts/run_pgn_pipeline.py

### Download Results

In [ ]:
from google.colab import files
!zip -r results_pgn.zip results_pgn/
files.download('results_pgn.zip')